In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score


LOAD + PREP DATA

In [10]:
df = pd.read_csv(r"D:\ML\SM Modeling\Data\Raw\incident_enriched.csv")

df['short_description'] = df['short_description'].fillna('')
df['close_notes'] = df['close_notes'].fillna('')
df['task_text'] = df['task_text'].fillna('')
df['close_code'] = df['close_code'].fillna('')
df['u_resolution_sub_category'] = df['u_resolution_sub_category'].fillna('')
df['u_fault_category'] = df['u_fault_category'].fillna('')

df["text"] = (
    df["short_description"] + " " +
    df["close_notes"] + " " +
    df["task_text"]
)

# 🔥 include resolution + subcategory
df["text_full"] = (
    df["text"] +
    " RESOLUTION_" + df["close_code"] +
    " SUBCAT_" + df["u_resolution_sub_category"]
)

df["u_fault_category"] = df["u_fault_category"].fillna("NO_FAULT")

# also treat empty string as NO_FAULT
df.loc[df["u_fault_category"].str.strip()=="", "u_fault_category"] = "NO_FAULT"

print(df["u_fault_category"].value_counts().head(10))

print("Rows:", df.shape)
print("Unique fault categories:", df["u_fault_category"].nunique())


u_fault_category
NO_FAULT                                      69288
RWT1 (Engineer Visit - Right When Tested )     7142
Customer Caused Damage                         5387
Third Party Damage                             3103
Criminal Damage                                 295
Road Traffic Collision                          268
Contractor Damage                               217
Failed Visit Charge                               5
Name: count, dtype: int64
Rows: (85705, 17)
Unique fault categories: 8


REMOVE RARE FAULT CATEGORIES

In [11]:
counts = df["u_fault_category"].value_counts()

valid = counts[counts >= 20].index
df = df[df["u_fault_category"].isin(valid)].reset_index(drop=True)

print("After cleaning:", df.shape)
print("Remaining fault categories:", df["u_fault_category"].nunique())


After cleaning: (85700, 17)
Remaining fault categories: 7


SPLIT

In [12]:
X = df["text_full"]
y = df["u_fault_category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(len(X_train), len(X_test))


68560 17140


TFIDF

In [13]:
tfidf = TfidfVectorizer(
    max_features=80000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


SVD

In [14]:
svd = TruncatedSVD(n_components=500, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)


TRAIN MODEL

In [15]:
model_fault = SGDClassifier(
    loss='log_loss',
    max_iter=3000,
    n_jobs=-1,
    class_weight='balanced'
)

model_fault.fit(X_train_svd, y_train)

pred = model_fault.predict(X_test_svd)
acc = accuracy_score(y_test, pred)

print("\n🎯 Fault Category Accuracy:", round(acc*100,2), "%")



🎯 Fault Category Accuracy: 89.18 %


In [16]:
import joblib
import os

save_dir = r"D:\ML\SM Modeling\AI_ENGINE_V3\models"
os.makedirs(save_dir, exist_ok=True)

In [17]:
joblib.dump(model_fault, save_dir + r"\fault_model.pkl")
joblib.dump(tfidf, save_dir + r"\fault_tfidf.pkl")
joblib.dump(svd, save_dir + r"\fault_svd.pkl")

print("Fault model saved successfully")


Fault model saved successfully
